In [37]:
import polars as pl
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [39]:
# ----------------------------------------------------------------------
# 1. Загрузка данных
# ----------------------------------------------------------------------
train = pl.read_parquet('data/train_main_features.parquet')
test = pl.read_parquet('data/test_main_features.parquet')
target = pl.read_parquet('data/train_target.parquet')

In [40]:
# ----------------------------------------------------------------------
# 2. Предобработка
# ----------------------------------------------------------------------
# Определяем категориальные признаки (начинаются с "cat_feature")
cat_features = [col for col in train.columns if col.startswith("cat_feature")]

# Преобразуем категориальные признаки в int32 (CatBoost требует int или str)
train = train.with_columns(pl.col(cat_features).cast(pl.Int32))
test = test.with_columns(pl.col(cat_features).cast(pl.Int32))

# Отделяем customer_id от признаков
train_ids = train['customer_id']
train_features = train.drop('customer_id')
test_ids = test['customer_id']
test_features = test.drop('customer_id')

# Целевые переменные (исключаем customer_id)
target_cols = [col for col in target.columns if col.startswith("target")]
target_data = target.select(target_cols).to_pandas()

In [42]:
# ----------------------------------------------------------------------
# 3. Разделение на обучающую и валидационную выборки
# ----------------------------------------------------------------------
X_train, X_val, y_train, y_val = train_test_split(
    train_features.to_pandas(),
    target_data,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

In [43]:
# ----------------------------------------------------------------------
# 4. Обучение отдельных бинарных моделей для каждого класса
# ----------------------------------------------------------------------
print("\nОбучение моделей (one-vs-rest) для 41 класса...")
models = {}
val_auc = {}

for i, target_col in enumerate(target_cols):
    print(f"\nОбучаем модель {i+1}/{len(target_cols)}: {target_col}")
    
    # Создаём Pool с указанием категориальных признаков
    train_pool = Pool(
        X_train,
        label=y_train[target_col],
        cat_features=cat_features
    )
    val_pool = Pool(
        X_val,
        label=y_val[target_col],
        cat_features=cat_features
    )
    
    # Инициализация модели
    model = CatBoostClassifier(
        iterations=200,               # увеличенное число итераций
        depth=6,                      # глубина деревьев
        learning_rate=0.05,           # маленький шаг
        loss_function='Logloss',      # бинарная классификация
        eval_metric='AUC',            # метрика для ранней остановки
        l2_leaf_reg=3,                # регуляризация
        random_seed=42,
        verbose=False,
        early_stopping_rounds=20,
        task_type='CPU',              # можно 'GPU', если есть
        thread_count=4                # количество потоков
    )
    
    # Обучение
    model.fit(
        train_pool,
        eval_set=val_pool,
        plot=False
    )
    
    # Предсказание на валидации и вычисление ROC-AUC
    pred_val = model.predict_proba(val_pool)[:, 1]
    auc = roc_auc_score(y_val[target_col], pred_val)
    val_auc[target_col] = auc
    models[target_col] = model
    
    print(f"  ROC-AUC на валидации: {auc:.4f}")



Обучение моделей (one-vs-rest) для 41 класса...

Обучаем модель 1/41: target_1_1
  ROC-AUC на валидации: 0.8951

Обучаем модель 2/41: target_1_2
  ROC-AUC на валидации: 0.7878

Обучаем модель 3/41: target_1_3
  ROC-AUC на валидации: 0.8552

Обучаем модель 4/41: target_1_4
  ROC-AUC на валидации: 0.8104

Обучаем модель 5/41: target_1_5
  ROC-AUC на валидации: 0.8446

Обучаем модель 6/41: target_2_1
  ROC-AUC на валидации: 0.8049

Обучаем модель 7/41: target_2_2
  ROC-AUC на валидации: 0.9201

Обучаем модель 8/41: target_2_3
  ROC-AUC на валидации: 0.7671

Обучаем модель 9/41: target_2_4
  ROC-AUC на валидации: 0.7251

Обучаем модель 10/41: target_2_5
  ROC-AUC на валидации: 0.6823

Обучаем модель 11/41: target_2_6
  ROC-AUC на валидации: 0.7168

Обучаем модель 12/41: target_2_7
  ROC-AUC на валидации: 0.8404

Обучаем модель 13/41: target_2_8
  ROC-AUC на валидации: 0.9689

Обучаем модель 14/41: target_3_1
  ROC-AUC на валидации: 0.6739

Обучаем модель 15/41: target_3_2
  ROC-AUC на вал

In [44]:
# ----------------------------------------------------------------------
# 5. Предсказание на тестовых данных
# ----------------------------------------------------------------------
print("\nФормирование предсказаний для тестовой выборки...")
test_pool = Pool(test_features.to_pandas(), cat_features=cat_features)

# Получаем вероятности для всех классов
predictions = []
for target_col in target_cols:
    model = models[target_col]
    proba = model.predict_proba(test_pool)[:, 1]
    predictions.append(proba)

# Транспонируем и создаём DataFrame
predictions = np.column_stack(predictions)
predict_cols = [f"predict_{col.replace('target_', '')}" for col in target_cols]
pred_df = pl.DataFrame(predictions, schema=predict_cols)


Формирование предсказаний для тестовой выборки...


In [45]:
# ----------------------------------------------------------------------
# 6. Формирование финального сабмита
# ----------------------------------------------------------------------
submit = pl.DataFrame({'customer_id': test_ids}).hstack(pred_df)
submit.write_parquet("data/submit.parquet")
print("\nСабмит сохранён в 'data/submit.parquet'")

# Вывод среднего ROC-AUC на валидации
mean_auc = np.mean(list(val_auc.values()))
print(f"\nСредний ROC-AUC (macro) на валидации: {mean_auc:.4f}")


Сабмит сохранён в 'data/submit.parquet'

Средний ROC-AUC (macro) на валидации: 0.8049
